[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ManzarIMalik/in2stem-placement/blob/main/Intro_to_Data_Science_with_Python.ipynb)

# What is Data Science

Data science is the business of turning raw data into decisions. It borrows from three places at once: **statistics** (what can this data honestly tell me?), **computer science** (how do I handle it at scale?), and **domain knowledge** (what would actually be surprising here?).

That third one is the one people skip, and it is the one that matters most. A chart is easy to make. Knowing whether it means anything is the hard part - and it is the skill this notebook is really about.

By the end you will have taken a real dataset of 3,900 customer purchases and gone the whole way: loading it, cleaning it, exploring it, visualising it, and then - the important bit - working out which of your findings are real and which are noise wearing a convincing hat.

**A warning up front.** This notebook will show you a chart that looks like a clear, obvious insight. It will then prove to you that the chart is meaningless. That is not a trick for its own sake: producing confident nonsense is the single most common way data science goes wrong, and the only defence is knowing what it looks like from the inside.

## Why Python for Data Science

* **It is readable.** You spend your time on the problem, not on the language.
* **The libraries are already there.** NumPy and Pandas do the heavy lifting; Matplotlib draws the pictures; scikit-learn does the machine learning. All of them are in Colab already.
* **Everyone else uses it.** When you get stuck at 2am, somebody has already asked your question.

The three libraries in this notebook stack on top of each other. **NumPy** is the foundation - fast arrays of numbers. **Pandas** is built on NumPy and adds labels, so your columns have names instead of index positions. **Matplotlib** draws whatever the other two produce. Learn them in that order and each one explains the next.

## The Data Science Workflow

Nearly every project walks through the same steps:

1. **Collect** the data - from a file, an API, a database, a scraper.
2. **Clean** it - handle missing values, wrong types, duplicates. This genuinely is most of the job.
3. **Explore** it (EDA) - look at it before you model it. Find the shape, the gaps, and the surprises.
4. **Model** it - build something that predicts or classifies.
5. **Evaluate** it - and be honest, especially when the answer is "this doesn't work".
6. **Communicate** it - a finding nobody understands is not a finding.

This notebook follows those steps in order, including step 5 - where we will find out that our data cannot answer the question we ask of it, which is a real result and worth knowing how to spot.

## Other Tools for Data Science

Python is not the only way to do this, and for a lot of business reporting it is not even the fastest way:

* [Power BI](https://www.microsoft.com/en-us/power-platform/products/power-bi) - Microsoft's dashboard tool, everywhere in industry.
* [Tableau](https://www.tableau.com/) - the other big dashboard tool.
* [Google Looker Studio](https://lookerstudio.google.com/) - free, browser-based, connects straight to Google Sheets.
* **Excel / Google Sheets** - genuinely fine for small data. Do not let anyone make you feel bad about it.

The reason to reach for Python instead: **repeatability**. A notebook re-runs from top to bottom and produces the same answer. A spreadsheet someone hand-edited for three months does not.

In [ ]:
# Colab already ships with everything this notebook needs - nothing to install.
import sys
import numpy as np
import pandas as pd
import matplotlib
import sklearn

print("Python version:      ", sys.version.split()[0])
print("NumPy version:       ", np.__version__)
print("pandas version:      ", pd.__version__)
print("Matplotlib version:  ", matplotlib.__version__)
print("scikit-learn version:", sklearn.__version__)
print("Ready to go.")

# Numerical Python (NumPy)

NumPy is the foundation the rest of the scientific Python world is built on. Its one big idea is the **`ndarray`**: a grid of numbers, all of the same type, stored in one continuous block of memory.

That "all the same type" restriction sounds like a limitation. It is actually the whole point - it is what makes NumPy fast, and we will measure exactly how much in a moment.

In [ ]:
import numpy as np

# A 1-dimensional array - like a list, but not a list.
a = np.array([1, 2, 3, 4, 5])
print("1D array:", a)
print("type:    ", type(a))

# A 2-dimensional array - a list of lists becomes a grid.
b = np.array([[1, 2, 3],
              [4, 5, 6]])
print()
print("2D array:")
print(b)

Typing out every number gets old fast. NumPy has constructors for the arrays you actually need.

In [ ]:
# Filled with a constant
print("zeros((2, 3)):")
print(np.zeros((2, 3)))

print("\nones((2, 4)):")
print(np.ones((2, 4)))

# arange(start, stop, step) - like range(), but an array. Stop is NOT included.
print("\narange(0, 10, 2):", np.arange(0, 10, 2))

# linspace(start, stop, n) - n evenly spaced points. Stop IS included.
# This trips everyone up: arange excludes the end, linspace includes it.
print("linspace(0, 1, 5):", np.linspace(0, 1, 5))

# Random numbers, with a seed so you get the same "random" numbers every run.
rng = np.random.default_rng(seed=42)
print("\nrandom integers:", rng.integers(1, 7, size=5))   # five dice rolls

### Array Attributes

Every array can describe itself. When something breaks in NumPy, the answer is almost always in `.shape`.

* `ndim` - how many dimensions.
* `shape` - the size along each dimension, as a tuple.
* `size` - the total number of elements.
* `dtype` - what type the elements are.

In [ ]:
arr = np.array([[1, 2, 3],
                [4, 5, 6]])

print("Number of dimensions:", arr.ndim)    # 2
print("Shape of the array:  ", arr.shape)   # (2, 3) -> 2 rows, 3 columns
print("Size of the array:   ", arr.size)    # 6 -> 2 * 3
print("Data type:           ", arr.dtype)   # int64

# Watch what "all the same type" means in practice:
mixed = np.array([1, 2, 3.5])
print()
print("np.array([1, 2, 3.5]) ->", mixed, "with dtype", mixed.dtype)
print("The integers were quietly converted to floats to keep one type for all.")

## Why NumPy Is Fast

Textbooks assert that NumPy is faster than plain Python lists. Let's not take that on trust - let's time it.

The key idea is **vectorization**. When you write `a * 2` on a NumPy array, the loop still happens, but it happens down in compiled C code over a tight block of memory, instead of in the Python interpreter one object at a time.

In [ ]:
import time

size = 1_000_000
python_list = list(range(size))
numpy_array = np.arange(size)

# The Python way: loop, one element at a time.
start = time.perf_counter()
result_list = [x * 2 for x in python_list]
python_time = time.perf_counter() - start

# The NumPy way: describe what you want to the whole array at once.
start = time.perf_counter()
result_array = numpy_array * 2
numpy_time = time.perf_counter() - start

print(f"Python list: {python_time * 1000:7.2f} ms")
print(f"NumPy array: {numpy_time * 1000:7.2f} ms")
print(f"NumPy is roughly {python_time / numpy_time:.0f}x faster here.")
print()
print("Same answer either way:", result_list[:3], "vs", result_array[:3])

The exact multiple changes every run - it depends on what else the machine is doing - but the order of magnitude is the point. This is why the whole ecosystem sits on NumPy.

There is a second reason, and it matters just as much for readability: `numpy_array * 2` says *what you want*. The list comprehension says *how to get it*. Once you are doing real work, the first one is much harder to get wrong.

## Indexing, Slicing, and Boolean Masks

Indexing an array works like indexing a list, with one big addition: you can index with a **condition**. This is how nearly all real filtering gets done, in both NumPy and Pandas, so it is worth getting comfortable with here where the arrays are small.

In [ ]:
data = np.array([12, 45, 7, 88, 23, 61, 4])

# Ordinary indexing and slicing, exactly like lists
print("First element:  ", data[0])
print("Last element:   ", data[-1])
print("Slice [1:4]:    ", data[1:4])

# A comparison on an array gives you an array of True/False - one per element.
mask = data > 20
print()
print("data > 20 ->", mask)

# Feed that back in as an index, and you get only the elements where it was True.
print("data[data > 20] ->", data[data > 20])

# Which is why counting is just a sum - True counts as 1.
print("How many are over 20?", (data > 20).sum())

For 2D arrays you index with `[row, column]`. A `:` means "everything along this dimension".

In [ ]:
grid = np.array([[1,  2,  3,  4],
                 [5,  6,  7,  8],
                 [9, 10, 11, 12]])

print("Single element [1, 2]:", grid[1, 2])   # row 1, column 2 -> 7
print("Whole row [0, :]:     ", grid[0, :])   # -> [1 2 3 4]
print("Whole column [:, 1]:  ", grid[:, 1])   # -> [2 6 10]
print("Sub-grid [0:2, 1:3]:")
print(grid[0:2, 1:3])

## Aggregations and `axis`

`sum`, `mean`, `min`, `max` and friends collapse an array down to a summary. On a 2D array you get to choose *which direction* to collapse, using `axis`.

The rule that makes `axis` finally click: **`axis` is the dimension that disappears.**

* `axis=0` collapses the rows, leaving one value per column.
* `axis=1` collapses the columns, leaving one value per row.

In [ ]:
grid = np.array([[1,  2,  3,  4],
                 [5,  6,  7,  8],
                 [9, 10, 11, 12]])

print("grid.shape:", grid.shape)             # (3, 4)
print()
print("Everything:      grid.sum()       ->", grid.sum())
print("Down the rows:   grid.sum(axis=0) ->", grid.sum(axis=0), " shape", grid.sum(axis=0).shape)
print("Across the cols: grid.sum(axis=1) ->", grid.sum(axis=1), " shape", grid.sum(axis=1).shape)
print()
print("Note the shapes: (3, 4) with axis=0 gone leaves (4,). With axis=1 gone leaves (3,).")
print()
print("mean:", grid.mean(), " min:", grid.min(), " max:", grid.max())

**Try it yourself:** `scores` below is 4 students by 3 tests. Work out (a) each student's average across their tests, and (b) each test's average across the students. One of them is `axis=0` and one is `axis=1` - use the "the axis disappears" rule to decide which, then check the shape of your answer to confirm.

In [ ]:
scores = np.array([[80, 72, 91],    # student 0
                   [65, 70, 68],    # student 1
                   [90, 95, 99],    # student 2
                   [50, 61, 58]])   # student 3

print("shape:", scores.shape, "-> 4 students, 3 tests")

# TODO: average per student. Should give 4 numbers.
# print("Per student:", scores.mean(axis=?))

# TODO: average per test. Should give 3 numbers.
# print("Per test:   ", scores.mean(axis=?))

**The answer:** `axis=1` gives the per-student averages, `axis=0` gives the per-test averages.

```
scores.mean(axis=1)  ->  [81.0, 67.67, 94.67, 56.33]   4 numbers, one per student
scores.mean(axis=0)  ->  [71.25, 74.5, 79.0]           3 numbers, one per test
```

If you had to guess, you are not alone - this is the single most-confused thing in NumPy, and the confusion comes from reading `axis` as "the direction I want to average along". Try the rule from above instead: **axis is the dimension that disappears.**

`scores` has shape `(4, 3)` - 4 students, 3 tests. Averaging with `axis=1` destroys dimension 1, the one of size 3 (the tests), leaving 4 numbers: one per student. Averaging with `axis=0` destroys dimension 0, the one of size 4 (the students), leaving 3 numbers: one per test.

**Check the shape of what comes back, every time.** Four numbers means you averaged over tests; three means you averaged over students. Getting this backwards will not raise an error - it will hand you a confident, plausible, wrong answer, which is a far worse outcome than a crash. Note that this only works as a check because 4 and 3 are different: on a square array both give the same shape and nothing will save you but reading it properly.

# Pandas

NumPy is fast but anonymous - `grid[:, 1]` gives you the second column and tells you nothing about what it means. Real data has names, mixed types, and gaps.

**Pandas** is NumPy with labels bolted on. It gives you two structures:

* **`Series`** - a 1D column of values with an index.
* **`DataFrame`** - a 2D table of Series sharing an index. A spreadsheet, or a SQL table.

Underneath, it is still NumPy doing the arithmetic - which is why everything you just learned about vectorization and boolean masks carries straight over.

In [ ]:
import pandas as pd
import numpy as np

# A Series from a list. Note np.nan - pandas' marker for "missing".
s = pd.Series([1, 3, 5, np.nan, 6, 8])
print(s)
print()
print("dtype is float64, not int64 - because np.nan is a float, so the whole column became float.")

The index is the thing that makes a Series more than an array. By default it is 0, 1, 2..., but it can be anything - and then you look values up by *name*.

In [ ]:
s = pd.Series([0.25, 0.5, 0.75, 1.0],
              index=['a', 'b', 'c', 'd'])
print(s)
print()

# Look up by label
print("s['b'] ->", s['b'])

# Slice by label - and note the surprise: unlike Python slicing,
# label slicing INCLUDES the endpoint.
print("s['a':'c'] ->", list(s['a':'c']), "  <- 'c' is included!")

# Slice by position - excludes the endpoint, like normal Python.
print("s[0:2]    ->", list(s[0:2]), "        <- position 2 is not")

A **DataFrame** is what you will spend nearly all your time in. Each column is a Series, and columns can have different types.

In [ ]:
data = {'Name': ['Alice', 'Bob', 'Charlie', 'David'],
        'Age': [25, 30, 35, 40],
        'City': ['New York', 'Los Angeles', 'Chicago', 'Houston']}

df_demo = pd.DataFrame(data)
print(df_demo)
print()
print("dtypes - each column has its own type:")
print(df_demo.dtypes)

## Getting Real Data

Enough toy examples. We will use the **Customer Shopping Trends** dataset: 3,900 purchases with age, gender, category, amount spent, review rating, and more.

The cell below downloads it directly, so this notebook runs top to bottom with nothing to set up. In the real world your data arrives as a file someone emails you, a database query, or an API - but it always ends up in the same place: a DataFrame.

In [ ]:
import urllib.request

url = "https://huggingface.co/datasets/vinaydanidhariya/Shopping-trends/resolve/main/csv_shopping_trends"
urllib.request.urlretrieve(url, "shopping_trends.csv")

df = pd.read_csv("shopping_trends.csv")
print("Downloaded. Shape:", df.shape, "-> 3900 rows, 18 columns")

## The First Look

Before anything else, four questions: **How big is it? What is in it? What does it look like? What are the numbers doing?**

There are four methods that answer those, and running them is the first thing you should do with any dataset, every time.

In [ ]:
# .shape - rows and columns
print("Shape:", df.shape)
print()

# .columns - what have we got?
print("Columns:")
for c in df.columns:
    print("  -", c)

In [ ]:
# .head() - the first few rows. The single most-used method in pandas.
df.head()

In [ ]:
# .info() - the structural summary: types, and how many non-null values per column.
df.info()

Read that `.info()` output carefully, because it is quietly telling us something important.

**Non-Null Count is 3900 for every single column**, and there are 3900 rows. So there are **no missing values anywhere**. That is unusual for real data - and it is our first hint that this dataset may be synthetic rather than collected. Hold that thought.

Also note the `Dtype` column. `object` means text. If a column you expect to be numeric shows up as `object`, something is wrong - usually a stray `"N/A"` or a `£` sign hiding in there.

In [ ]:
# .describe() - the statistical summary of the numeric columns.
df.describe()

`.describe()` repays slow reading. Look at **Purchase Amount (USD)**: min 20, max 100, mean about 60, and the quartiles sit at roughly 39 / 60 / 81 - almost perfectly evenly spread. Real spending is not shaped like that. Real spending is lumpy and right-skewed: lots of small purchases, a few big ones.

Second hint. Let's keep going.

## Cleaning the Data

The textbook line is that cleaning is 80% of the job, and the textbook move at this point is:

```python
new_df = df.dropna()
```

Let's actually check whether that does anything here, rather than performing the ritual and moving on.

In [ ]:
print("Rows before dropna():", len(df))
print("Rows after  dropna():", len(df.dropna()))
print("Rows removed:        ", len(df) - len(df.dropna()))
print()
print("Total missing values in the entire dataset:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

**`dropna()` removed nothing, because there was nothing to remove.**

This matters more than it looks. Calling `dropna()` here and moving on would let you *feel* like you cleaned the data while doing precisely nothing. Cleaning is not a spell you cast - it is a response to specific problems you have found. If you have not looked for the problems, you have not cleaned anything.

So what does real cleaning look like when there *is* something wrong? Here is the toolkit, demonstrated on a small broken table:

In [ ]:
# A deliberately messy table - the kind of thing real data actually looks like.
messy = pd.DataFrame({
    'name':   ['Alice', 'Bob', 'Charlie', 'Bob', 'Eve'],
    'age':    [25, np.nan, 35, np.nan, 29],
    'salary': ['50000', '60000', 'unknown', '60000', '55000'],
})
print("Original:")
print(messy)

# 1. Find the missing values before deciding anything.
print("\nMissing per column:")
print(messy.isnull().sum())

# 2. Drop exact duplicate rows (Bob appears twice).
messy = messy.drop_duplicates()

# 3. Fill missing numbers rather than dropping the row - dropping loses the other columns too.
messy['age'] = messy['age'].fillna(messy['age'].median())

# 4. Force a text column to numeric. errors='coerce' turns 'unknown' into NaN
#    instead of raising - then we can decide what to do with it.
messy['salary'] = pd.to_numeric(messy['salary'], errors='coerce')
messy['salary'] = messy['salary'].fillna(messy['salary'].median())

print("\nCleaned:")
print(messy)
print("\ndtypes are now numeric, so we can actually do maths:")
print(messy.dtypes)

The decision in step 3 is the interesting one. `dropna()` throws away the **whole row** because of one missing cell - so a missing age costs you that person's salary too. Filling with the median keeps the row at the cost of inventing a value. Neither is automatically right, and that judgement call is the actual work.

Our shopping data needs none of this. That is a fact about the data, not a sign we skipped a step.

## Selecting and Filtering

Everything you learned about NumPy boolean masks works here, except now you can name the columns.

In [ ]:
# One column -> a Series
print("One column:", type(df['Age']).__name__)

# A list of columns -> a DataFrame. Note the double brackets.
print("Two columns:", type(df[['Age', 'Gender']]).__name__)
print()

# Boolean filtering - exactly the NumPy idea, with names.
big_spenders = df[df['Purchase Amount (USD)'] > 90]
print("Purchases over $90:", len(big_spenders))

# Combine conditions with & (and) / | (or). The brackets around each are REQUIRED.
young_big = df[(df['Age'] < 30) & (df['Purchase Amount (USD)'] > 90)]
print("Under 30 AND spent over $90:", len(young_big))

print()
print(young_big[['Age', 'Gender', 'Category', 'Purchase Amount (USD)']].head())

`.loc` and `.iloc` are how you get at rows. The distinction catches everyone out once:

* **`.loc[]`** - by **label** (and, as we saw with Series, the end is included).
* **`.iloc[]`** - by **integer position** (the end is excluded, like normal Python).

In [ ]:
# .iloc - by position
print("First row, by position:")
print(df.iloc[0][['Age', 'Gender', 'Category']])

print("\nRows 0-2, two columns, by position:")
print(df.iloc[0:3, [1, 2]])

# .loc - by label. Our index is 0,1,2... so the labels look like positions -
# but watch the endpoint behave differently.
print("\n.iloc[0:3] gives", len(df.iloc[0:3]), "rows (end excluded)")
print(".loc[0:3]  gives", len(df.loc[0:3]), "rows (end included)")

## Grouping and Aggregating

`groupby` is the workhorse of data analysis. It is one idea in three steps - **split, apply, combine**:

1. **Split** the rows into groups by some column.
2. **Apply** a summary to each group.
3. **Combine** the answers into a new table.

"Average purchase amount per category" is: split by `Category`, apply `mean` to `Purchase Amount (USD)`, combine.

In [ ]:
# Split by Category, apply mean, combine.
print("Average purchase by category:")
print(df.groupby('Category')['Purchase Amount (USD)'].mean().round(2))

print("\nHow many purchases per category:")
print(df['Category'].value_counts())

# .agg() gets you several summaries at once - far more useful than mean alone,
# because count and std tell you how much to trust the mean.
print("\nSeveral statistics at once:")
print(df.groupby('Category')['Purchase Amount (USD)'].agg(['count', 'mean', 'std']).round(2))

Park that last table for a moment. The means are all hovering around 57-60 with a standard deviation of about 24. We will come back to what that means, and it is the most important thing in this notebook.

**Try it yourself:** use `groupby` to find the average `Review Rating` for each `Season`. Then use `.agg(['count', 'mean', 'std'])` instead of `.mean()` - does knowing the count and spread change how confident you feel about the ranking?

In [ ]:
# TODO: average Review Rating per Season
# print(df.groupby(?)[?].mean())

# TODO: now with count and std as well. Are the differences big compared to the spread?
# print(df.groupby(?)[?].agg(['count', 'mean', 'std']))

**The answer**, and it is worth more than the syntax.

```
        count   mean    std
Fall      975  3.730  0.712
Spring    999  3.791  0.726
Summer    955  3.726  0.715
Winter    971  3.752  0.711
```

`df.groupby('Season')['Review Rating'].mean()` gives you the first column, and `.agg(['count', 'mean', 'std'])` gives you all three.

Now the second question, which is the real one. **Is the difference big compared to the spread?**

* Gap between the best season and the worst: **0.065 stars**.
* Typical spread of ratings *within* any one season: **0.716 stars**.

The gap is about **one eleventh** of the noise. Spring is not a better season to shop in. You are looking at four numbers that are, for all practical purposes, the same number - and if you plotted them on a bar chart with a zoomed axis, Spring would tower over Summer and you would have a slide for Monday's meeting.

This is habit 3 from the chart section, arriving early: **compare the gap between groups to the spread within groups**. It costs one extra word - `.agg(['count', 'mean', 'std'])` instead of `.mean()` - and it is the difference between seeing four numbers and understanding them.

# Visualising with Matplotlib

A table of numbers hides its shape. A chart shows it instantly - which is exactly why charts are also the easiest way to mislead people, including yourself.

The rule of thumb for picking one:

| You want to show | Use |
|---|---|
| The distribution of one numeric column | **Histogram** |
| Comparing a value across categories | **Bar chart** |
| Parts of a whole (a handful of categories) | **Pie chart** |
| The relationship between two numeric columns | **Scatter plot** |
| Something changing over time | **Line chart** |

In [ ]:
import matplotlib.pyplot as plt

# A histogram buckets a numeric column and counts what lands in each bucket.
plt.figure(figsize=(8, 4))
plt.hist(df['Age'], bins=20, color='skyblue', edgecolor='black')
plt.title('Age Distribution')
plt.xlabel('Age')
plt.ylabel('Number of customers')
plt.show()

Nearly flat from 18 to 70. Ages in a real customer base are never flat - they cluster.

That is our third hint that this data was generated rather than gathered, and the histogram made it obvious in a way `.describe()` did not. This is what EDA is *for*.

In [ ]:
# A bar chart compares a value across categories.
plt.figure(figsize=(6, 4))
df['Gender'].value_counts().plot(kind='bar', color=['steelblue', 'lightcoral'])
plt.title('Gender Distribution')
plt.xlabel('Gender')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.show()

print(df['Gender'].value_counts())
print()
print("Male share:", round((df['Gender'] == 'Male').mean() * 100, 1), "%")

In [ ]:
# Top 10 items - value_counts() sorts for you, so .head(10) takes the top 10.
plt.figure(figsize=(9, 4))
df['Item Purchased'].value_counts().head(10).plot(kind='bar', color='mediumslateblue')
plt.title('Top 10 Purchased Items')
plt.xlabel('Item')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

Look at the y-axis. The most popular item appears 171 times and the tenth 161 - out of 3,900 purchases across 25 items. If every item were equally likely you would expect about 156 each.

So this "Top 10" is a ranking of essentially nothing. The bar chart still draws a confident staircase, because that is what bar charts do. **A chart will always find a pattern; it is your job to decide whether the pattern is real.**

Which brings us to the main event.

# When a Chart Lies

Here is a chart of the kind you will see in a hundred tutorials. It is the one the original version of this notebook drew.

Run the cell, look at the chart, and **write your answer down before you scroll** - in the cell below, or on paper, but write it somewhere. One sentence: what would you tell your manager?

This matters more than it sounds. In a moment this notebook is going to take that chart apart, and once you have read the explanation you will be certain you saw it coming. Everybody is. The only way to find out whether you actually did is to have committed to something first.

In [ ]:
# Average purchase amount by category, sorted - a classic-looking chart.
plt.figure(figsize=(8, 4))
df.groupby('Category')['Purchase Amount (USD)'].mean().sort_values().plot(
    kind='barh', color='lightgreen', edgecolor='black')
plt.title('Average Purchase Amount by Category')
plt.xlabel('Avg Purchase Amount (USD)')
plt.ylabel('Category')
plt.show()

**Write your answer here before reading on.** What does this chart tell you, and what would you do about it?

Your answer: 

The story writes itself: *"Footwear customers spend the most, Outerwear the least. We should push Footwear."*

That conclusion is worthless. Here is why.

**Clue 1: look at the x-axis.** It does not start at zero, and Matplotlib zoomed in to fill the space. The entire visual difference between the longest and shortest bar is about **$3** - on purchases that range from $20 to $100.

**Clue 2: `.sort_values()` guarantees a staircase.** Sorting *anything* produces a ranking. Sort four random numbers and one of them is the biggest. The chart cannot tell you whether the order means something, because the sort imposes the order regardless.

**Clue 3: we already saw the standard deviation was ~24.** The gaps between the groups (about $3) are tiny next to the variation inside each group (about $24).

Let's test it properly instead of squinting at it.

In [ ]:
# The shuffle test: if Category has NO effect on spending, then randomly
# reassigning the category labels should produce a chart just as dramatic.
observed = df.groupby('Category')['Purchase Amount (USD)'].mean()
observed_spread = observed.max() - observed.min()

print("Observed gap between highest and lowest category: $%.2f" % observed_spread)
print()

rng = np.random.default_rng(seed=0)
fake_spreads = []
for _ in range(2000):
    shuffled = rng.permutation(df['Category'].values)          # labels randomised
    means = df.groupby(shuffled)['Purchase Amount (USD)'].mean()
    fake_spreads.append(means.max() - means.min())
fake_spreads = np.array(fake_spreads)

print("With the labels shuffled at random, 2000 times:")
print("  average gap: $%.2f" % fake_spreads.mean())
print("  gap was >= our 'real' one in %.1f%% of the random shuffles" % ((fake_spreads >= observed_spread).mean() * 100))

**Pure chance beats our "finding" about 10% of the time.** In other words, if Category made no difference whatsoever to how much people spend, we would still see a gap this big roughly one time in ten - just from the luck of who fell into which group.

That is nowhere near strong enough to act on. The chart was drawing noise, and the sorting made the noise look like a trend.

In [ ]:
# The same result, drawn. Where does our observed gap sit among the random ones?
plt.figure(figsize=(8, 4))
plt.hist(fake_spreads, bins=40, color='lightgray', edgecolor='black',
         label='Gaps from randomly shuffled labels')
plt.axvline(observed_spread, color='red', linewidth=2,
            label=f'Our "real" gap (${observed_spread:.2f})')
plt.title('Our finding vs. what pure chance produces')
plt.xlabel('Gap between highest and lowest category mean (USD)')
plt.ylabel('How often')
plt.legend()
plt.show()

print("Our red line sits inside the pile of random results, not out beyond it.")
print("If Category really mattered, the red line would be far off to the right.")

The red line is sitting comfortably inside the range of things that happen by accident. There is no finding here.

**The honest chart** shows the uncertainty instead of hiding it. Same data, error bars showing the spread, and an x-axis that starts at zero:

In [ ]:
stats = df.groupby('Category')['Purchase Amount (USD)'].agg(['mean', 'std', 'count'])

# The standard error tells us how much the mean itself would wobble
# if we collected the data again.
stats['stderr'] = stats['std'] / np.sqrt(stats['count'])

plt.figure(figsize=(8, 4))
plt.barh(stats.index, stats['mean'], xerr=stats['stderr'] * 1.96,   # 95% interval
         color='lightgreen', edgecolor='black', capsize=5)
plt.title('Average Purchase by Category (with 95% confidence intervals)')
plt.xlabel('Avg Purchase Amount (USD)')
plt.xlim(0, 100)          # start at zero - no free drama
plt.show()

print(stats.round(2))
print()
print("The error bars overlap and the bars are nearly identical. There is no story here,")
print("and this version of the chart is honest about that.")

Compare the two charts. **Same data, opposite conclusions.** The first one is not a mistake anyone made on purpose - it is just what you get from the default settings and a `.sort_values()` call.

Three habits that protect you:

1. **Start bar charts at zero.** A zoomed axis manufactures drama.
2. **Show the uncertainty.** If the error bars overlap, you do not have a finding.
3. **Compare the gap between groups to the spread within groups.** Small gap, big spread means nothing is there.

**Try it yourself:** the chart below does exactly the same thing for `Payment Method`. Run it, then adapt the shuffle test above to check it.

Before you run the shuffle test, commit to a guess and write it down. Chance beat the Category "finding" about 10% of the time. For Payment Method, will it be **more or less often than that** - and why? Look at the chart and think about how many groups there are and how big the gap is compared to the Category one.

In [ ]:
plt.figure(figsize=(8, 4))
df.groupby('Payment Method')['Purchase Amount (USD)'].mean().sort_values().plot(
    kind='bar', color='teal')
plt.title('Avg Purchase Amount by Payment Method - is this real?')
plt.xlabel('Payment Method')
plt.ylabel('Avg Purchase Amount')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# TODO: copy the shuffle test from above and run it on 'Payment Method'.
# How often does chance beat this "finding"?

**The answer.** Chance beats the Payment Method "finding" about **two thirds of the time** - roughly 67%, versus 10% for Category. If you guessed "more often", you were reading the chart correctly.

Two things make it even emptier than the Category chart:

* **The gap is smaller.** About $1.97 between the highest and lowest payment method, against $3.08 for Category - on purchases spanning $20 to $100.
* **There are more groups.** Six payment methods versus four categories. The more groups you split into, the more chances you give one of them to drift high or low by luck alone. More slices means more opportunities for a fluke, which is exactly why "we sliced the data and found something" should make you suspicious rather than excited.

Note that 67% is not a magic constant - re-run with a different seed and you will get something between about 66% and 68%. The shuffle test is itself a random process, so it wobbles. What matters is that it is nowhere near the 5% or so you would want before calling something real, not the precise digits.

# Finding What Is Actually There

We have established that the differences in spending are noise. So is there *anything* real in this dataset? Yes - and it turns up when you stop asking "what is the average of X" and start asking "how do these columns relate to each other?"

In [ ]:
# Do any of the numeric columns move together?
numeric_cols = ['Age', 'Purchase Amount (USD)', 'Review Rating', 'Previous Purchases']
print("Correlations between the numeric columns:")
print(df[numeric_cols].corr().round(3))

A correlation runs from -1 (perfect opposite) through 0 (no relationship) to +1 (perfect match). **Every off-diagonal number here is essentially zero.** Age has no bearing on spending. Rating has no bearing on spending. Nothing predicts anything.

Now the categorical columns - and here we finally hit something.

In [ ]:
# Discount Applied and Promo Code Used both split 1677 / 2223. Coincidence?
print(pd.crosstab(df['Discount Applied'], df['Promo Code Used']))
print()
print("Are the two columns literally identical, row for row?",
      (df['Discount Applied'] == df['Promo Code Used']).all())

**They are the same column twice.** Every single row agrees - the off-diagonal cells of that crosstab are zero.

This is a genuine finding, and a common one in real data. It matters because:

* Charting both is charting the same thing twice, and looks like corroboration when it is not.
* Feeding both to a model gives one fact double the weight.
* It usually means something upstream - here, "discount" and "promo code" are almost certainly two names for the same field.

One of them should be dropped. **This is what real cleaning looks like** - a specific response to a specific problem you went looking for.

In [ ]:
# And now the strangest thing in the dataset.
print("Gender vs Discount Applied:")
print(pd.crosstab(df['Gender'], df['Discount Applied']))
print()
print("Gender vs Subscription Status:")
print(pd.crosstab(df['Gender'], df['Subscription Status']))

Read those tables again. **Not one single female customer in 1,248 ever received a discount, or ever held a subscription.** Zero. Both columns, perfectly.

If this were real data, this would be the entire report - you would have found either serious discrimination or a broken pipeline, and either way you would stop and escalate immediately.

Here, combined with the flat age distribution, the suspiciously even spending, and the total absence of correlations, it confirms what we suspected: **this dataset is synthetic**. Someone generated it with a script, and that script had a bug in it.

Noticing that is the win. A "Top 10 Items" bar chart would never have told you.

# Modelling and Evaluation

The workflow at the top promised modelling and evaluation, so let's do it properly - including the part where we report an answer we did not want.

The question: **can we predict how much someone will spend, from everything else we know about them?**

Two rules before we start:

1. **Split the data.** Train on one part, test on a part the model has never seen. A model graded on data it memorised will always look brilliant. (`Intro_to_Computer_Science_and_AI.ipynb` covers this under overfitting.)
2. **Always have a baseline.** A number means nothing on its own. "85% accurate" is terrible if guessing gets you 84%.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import r2_score, mean_absolute_error

# Models need numbers, not text. get_dummies turns each category into 0/1 columns.
features = ['Age', 'Gender', 'Category', 'Season', 'Review Rating',
            'Subscription Status', 'Previous Purchases', 'Payment Method']
X = pd.get_dummies(df[features], drop_first=True)
y = df['Purchase Amount (USD)']

print("Feature matrix shape:", X.shape, "-> text columns became", X.shape[1], "numeric ones")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Training on", len(X_train), "rows, testing on", len(X_test), "unseen rows.")

In [ ]:
# The baseline: ignore every feature and always guess the average.
baseline = DummyRegressor(strategy='mean').fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

# The real model.
model = LinearRegression().fit(X_train, y_train)
model_pred = model.predict(X_test)

print("%-22s %8s %8s" % ("", "R^2", "MAE"))
print("%-22s %8.4f %8.2f" % ("Baseline (guess mean)", r2_score(y_test, baseline_pred), mean_absolute_error(y_test, baseline_pred)))
print("%-22s %8.4f %8.2f" % ("Linear Regression", r2_score(y_test, model_pred), mean_absolute_error(y_test, model_pred)))

**R² of roughly zero - and slightly negative.** R² is the share of the variation the model explains: 1.0 is perfect, 0.0 means "no better than always guessing the average", and negative means **actively worse than that**.

Our model, given age, gender, category, season, rating, subscription status, purchase history and payment method, is worse than a rule that ignores all of it and says "$60".

**This is not a bug, and it is not your fault.** It is the correct answer. We already proved the correlations were zero and the group differences were noise - so there was never any signal for the model to find. Everything in this notebook has been consistent.

What would be a bug is not noticing. Without the baseline, "MAE of about $21" sounds like a result you could put in a slide.

In [ ]:
# Would a fancier model rescue us? This is the reflex, so let's test the reflex.
from sklearn.ensemble import RandomForestRegressor

forest = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_train, y_train)
forest_pred = forest.predict(X_test)

print("%-22s %8s %8s" % ("", "R^2", "MAE"))
print("%-22s %8.4f %8.2f" % ("Baseline (guess mean)", r2_score(y_test, baseline_pred), mean_absolute_error(y_test, baseline_pred)))
print("%-22s %8.4f %8.2f" % ("Linear Regression", r2_score(y_test, model_pred), mean_absolute_error(y_test, model_pred)))
print("%-22s %8.4f %8.2f" % ("Random Forest", r2_score(y_test, forest_pred), mean_absolute_error(y_test, forest_pred)))

print()
print("Training-set R^2 for the forest: %.4f" % r2_score(y_train, forest.predict(X_train)))
print("Test-set R^2 for the forest:     %.4f" % r2_score(y_test, forest_pred))

The Random Forest is **worse than the linear model, which was worse than guessing** - while scoring far better on the training data than the test data.

That gap between training and test performance is **overfitting**, seen in the wild. With no signal to learn, the forest did the only thing left: it memorised the noise in the training set. That memorised noise is useless on new customers, so it actively hurts.

**The lesson.** A more powerful model cannot conjure signal that is not there. When your model will not beat the baseline, the answer is more or better *data* - not a bigger algorithm.

And "this data cannot answer this question" is a legitimate, valuable, publishable result. Saying so out loud is the job. The failure mode is torturing the data until it produces a chart that looks like the answer someone wanted - and by now you know exactly how easy that would have been.

# Homework

**Challenge 1: Finish the "Try it yourself" cells**
There are three above - the NumPy `axis` one, the `groupby` on Season, and the shuffle test on Payment Method. Do those first.

**Challenge 2: Find a dataset**
Go to [Kaggle](https://www.kaggle.com/datasets) and pick something you actually care about - sport, music, games, climate. Interest carries you through the boring parts.

To load it in Colab, either upload it via the file browser on the left (then read `/content/your_file.csv`), or find it on the [Hugging Face Hub](https://huggingface.co/datasets) and download it programmatically like we did above. The second way means your notebook re-runs anywhere.

**Challenge 3: Run the first look**
`.shape`, `.head()`, `.info()`, `.describe()`. Then write a markdown cell answering: how many rows? Any missing values? Anything in `.describe()` that looks impossible - a negative age, a rating above the maximum, a suspiciously round number?

**Challenge 4: Clean it - but only what needs cleaning**
Check for missing values and duplicates. If there are none, say so. Do not call `dropna()` as a ritual. If there *are* problems, fix them and write down what you decided and why.

**Challenge 5: Three charts, three questions**
Make a histogram, a bar chart, and one other. Each one has to answer a question you wrote down *before* you drew it.

**Challenge 6: Try to break your own finding**
Pick your most interesting result and attack it:
* Does the bar chart's axis start at zero? Redraw it if not.
* How big is the gap between groups compared to the standard deviation within them?
* Run the shuffle test. How often does chance beat you?

**Challenge 7: Report honestly**
Write a short markdown summary of what you found - **including anything that turned out to be noise**. A notebook that says "I expected X, tested it, and it was not there" is better work than one with five confident charts of nothing.

In [ ]:
# Challenge 2: load your dataset here.
# Option A - upload via the file browser on the left, then:
# my_df = pd.read_csv('/content/your_file.csv')

# Option B - straight from a URL (this way the notebook re-runs anywhere):
# my_df = pd.read_csv('https://huggingface.co/datasets/OWNER/NAME/resolve/main/file.csv')

# TODO: your code here.

In [ ]:
# Challenge 3: the first look. shape, head, info, describe.
# TODO: your code here.

In [ ]:
# Challenge 4: check for missing values and duplicates - clean only what is broken.
# TODO: your code here.

In [ ]:
# Challenge 5: three charts, each answering a question you wrote down first.
# TODO: your code here.

In [ ]:
# Challenge 6: try to break your best finding. Shuffle test, zeroed axis, error bars.
# TODO: your code here.

# Challenge 7 - write your honest summary here as markdown

*Replace this cell with what you found, what you expected, and what turned out to be noise.*

# What We Covered

* **NumPy** - the `ndarray`, and *why* it is fast rather than just that it is. Boolean masks and the `axis` rule that the collapsed dimension is the one that disappears.
* **Pandas** - Series and DataFrames, loading real data, `.loc` vs `.iloc`, filtering, and split-apply-combine with `groupby`.
* **Cleaning** - and the more important half: checking whether there is anything to clean, instead of calling `dropna()` as a reflex.
* **EDA** - the first look, and reading it properly. A flat age distribution and evenly-spread spending told us the data was synthetic before any model did.
* **Matplotlib** - histograms, bar charts, and how to choose between them.
* **When a chart lies** - a sorted bar chart with a zoomed axis turned a $3 difference into a story. The shuffle test showed chance produces that gap 10% of the time.
* **Real findings** - `Discount Applied` and `Promo Code Used` are the same column, and no female customer ever got a discount. Both are worth more than every average we computed.
* **Modelling** - split, baseline, evaluate. Our model lost to "always guess $60", the Random Forest lost by more, and reporting that is the correct outcome.

**The thread.** The tools are the easy part - you can look up `groupby` any time. The skill is scepticism about your own results. Any dataset will hand you a chart that looks like an insight. Data science is knowing which ones to believe.

**Where to go next**
* **Seaborn** - built on Matplotlib, statistical charts in one line, and it draws confidence intervals by default.
* **`Intro_to_Computer_Science_and_AI.ipynb`** - the theory under the modelling section: overfitting, train/test splits, and why more model is not more answer.
* **`Intro_to_NLP.ipynb`** - the same workflow where the data is text instead of numbers. That dataset *does* have signal, and the model beats its baseline by 15 points - a useful contrast with what happened here.
* **Statistics** - this notebook's shuffle test is a permutation test in disguise. Learning the formal versions (t-tests, ANOVA, confidence intervals) makes this rigorous instead of intuitive.